# Méthodes mathématiques de résolution

In [ ]:
from copy import deepcopy
import random
import sqlite3

import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

from tomography import random_discrete_image
from tomography.tomographic_reconstruction import back_projection
from tomography.tomographic_reconstruction import fourier_domain_reconstruction
from tomography.tomographic_reconstruction import iterative_reconstruction
from tomography.tomographic_reconstruction import radon_transform

pio.renderers.default = "browser"  # plotly_mimetype // browser

COLORSCALE: str = "Viridis"

In [ ]:
S = 65  # 55 65
np.random.seed(S)
random.seed(S)

image = random_discrete_image(100, 100, 20)

empty_img = deepcopy(image)
empty_img[empty_img == 0] = np.nan

fig = go.Figure(data=go.Heatmap(
    z=empty_img,
    colorscale=COLORSCALE,
    showscale=False,
)
)

fig.update_xaxes(scaleanchor='x',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks='outside',
                 showline=True,
                 showgrid=False,
                 scaleratio=1,
                 showticklabels=False)
fig.update_yaxes(scaleanchor='y',
                 linewidth=1,
                 linecolor='black',
                 mirror=True,
                 ticks="outside",
                 showline=True,
                 showgrid=False,
                 scaleratio=1,
                 showticklabels=False,
                 )

fig.update_layout(
    template='plotly_white',
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.025,
        bordercolor="black",
        borderwidth=1),
    title={"text": r"$\LARGE{\text{Simulation d'un densité osseuse}}$"},
)

# fig.write_image("images/2 fig0 lr.webp", scale=1.3 * 2, height=600, width=600)
# fig.write_image("images/2 fig0 hr.png", scale=13, height=600, width=600)

## Les méthodes

### Back Projection

In [ ]:
for i, n_angle in enumerate([10, 25, 100, 360]):
    print(n_angle)
    angles = np.linspace(0, 360, n_angle)
    sinogram = radon_transform(image, angles)

    reconstruction = back_projection(sinogram, angles)

    fig = go.Figure(data=go.Heatmap(
        z=reconstruction,
        colorscale=COLORSCALE,
        showscale=False,
    ))

    fig.update_layout(
        template='plotly_white',
        title={
            "text": r"$\LARGE{\text{Reconstruction de la densité osseuse}} \\ \large{\text{Rétroprojection - " + f"{n_angle} prises de vues" + "}} $"
        },
    )

    fig.update_xaxes(scaleanchor='x',
                     linewidth=1,
                     linecolor='black',
                     scaleratio=1,
                     showticklabels=False,
                     ticks=None, )
    fig.update_yaxes(scaleanchor='y',
                     linewidth=1,
                     linecolor='black',
                     showticklabels=False,
                     ticks=None,
                     )

    fig.write_image(f"images/2 fig1-{i} lr.webp", scale=1.3, height=600, width=600)
    fig.write_image(f"images/2 fig1-{i} hr.png", scale=13, height=600, width=600)


### Fourier domain reconstruction

In [ ]:
for i, n_angle in enumerate([10, 25, 100, 360]):
    print(n_angle)
    angles = np.linspace(0, 360, n_angle)
    sinogram = radon_transform(image, angles)

    reconstruction = fourier_domain_reconstruction(sinogram, angles)

    fig = go.Figure(data=go.Heatmap(
        z=reconstruction,
        colorscale=COLORSCALE,
        showscale=False,
    ))

    fig.update_layout(
        template='plotly_white',
        title={
            "text": r"$\LARGE{\text{Reconstruction de la densité osseuse}} \\ \large{\text{Reconstruction dans le domaine de Fourier - " + f"{n_angle} prises de vues" + "}} $"
        },
    )

    fig.update_xaxes(scaleanchor='x',
                     linewidth=1,
                     linecolor='black',
                     scaleratio=1,
                     showticklabels=False,
                     ticks=None, )
    fig.update_yaxes(scaleanchor='y',
                     linewidth=1,
                     linecolor='black',
                     showticklabels=False,
                     ticks=None,
                     )

    fig.write_image(f"images/2 fig2-{i} lr.webp", scale=1.3, height=600, width=600)
    fig.write_image(f"images/2 fig2-{i} hr.png", scale=13, height=600, width=600)


### Iterative reconstruction

In [ ]:
for i, n_angle in enumerate([10, 25, 100, 360]):
    print(n_angle)
    angles = np.linspace(0, 360, n_angle)
    sinogram = radon_transform(image, angles)

    reconstruction = iterative_reconstruction(sinogram, angles, n_iter=2)

    fig = go.Figure(data=go.Heatmap(
        z=reconstruction,
        colorscale=COLORSCALE,
        showscale=False,
    ))

    fig.update_layout(
        template='plotly_white',
        title={
            "text": r"$\LARGE{\text{Reconstruction de la densité osseuse}} \\ \large{\text{Reconstruction itérative (5 itérations) - " + f"{n_angle} prises de vues" + "}} $"
        },
    )

    fig.update_xaxes(scaleanchor='x',
                     linewidth=1,
                     linecolor='black',
                     scaleratio=1,
                     showticklabels=False,
                     ticks=None, )
    fig.update_yaxes(scaleanchor='y',
                     linewidth=1,
                     linecolor='black',
                     showticklabels=False,
                     ticks=None,
                     )

    fig.write_image(f"images/2 fig3-{i} lr.webp", scale=1.3, height=600, width=600)
    fig.write_image(f"images/2 fig3-{i} hr.png", scale=13, height=600, width=600)


## Nombre d'itérations pour la reconstruction interactive

In [ ]:
conn = sqlite3.connect(f"data/iterative_parameters.db", check_same_thread=False)
cursor = conn.cursor()

data = [*cursor.execute("SELECT * FROM results").fetchall()]

cursor.close()
conn.close()

x = np.array([*map(lambda c: c[0], data)])
y = np.array([*map(lambda c: c[1], data)])

In [ ]:
# Calculs statistiques avec NumPy
x_unique = np.arange(1, max(x) + 1)

# Listes des statistiques par catégorie
q1, q2, q3 = [], [], []
lowerfence, upperfence = [], []
mean, sd = [], []

# Calcul des quartiles et bornes par groupe de x
for xv in x_unique:
    y_group = y[x == xv]
    if len(y_group) == 0:
        q1.append(np.nan)
        q2.append(np.nan)
        q3.append(np.nan)
        lowerfence.append(np.nan)
        upperfence.append(np.nan)
        mean.append(np.nan)
        sd.append(np.nan)
        continue

    q1v = np.percentile(y_group, 25)
    q2v = np.percentile(y_group, 50)
    q3v = np.percentile(y_group, 75)
    iqr = q3v - q1v

    # Moustaches limitées par les valeurs réelles
    lower = max(np.min(y_group), q1v - 1.5 * iqr)
    upper = min(np.max(y_group), q3v + 1.5 * iqr)

    q1.append(q1v)
    q2.append(q2v)
    q3.append(q3v)
    lowerfence.append(lower)
    upperfence.append(upper)
    mean.append(np.mean(y_group))
    sd.append(np.std(y_group))

fig = go.Figure()

fig.add_trace(go.Box(
    x=x_unique,
    q1=q1,
    median=q2,
    q3=q3,
    lowerfence=lowerfence,
    upperfence=upperfence,
    mean=mean,
    sd=sd,
    boxmean='sd',
    line=dict(width=1.75),
))

fig.update_layout(
    template='plotly_white',
    title={
        "text": r"$\LARGE{\text{Comparaison de l'erreur en fonction du nombre d'itérations}} \\ \large{\text{Méthode itérative - 10 000 mesures par boîte - 360 prises de vues}}$"
    },
)

fig.update_xaxes(scaleanchor='x',
                 linewidth=1,
                 linecolor='black',
                 scaleratio=1,
                 showticklabels=True,
                 dtick=1,
                 )
fig.update_yaxes(scaleanchor='y',
                 linewidth=1,
                 linecolor='black',
                 showticklabels=True,
                 range=[0.0, 0.3]
                 )

fig.write_image(f"images/2 fig4 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image(f"images/2 fig4 hr.png", scale=13, height=600, width=1038)

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x_unique,
    y=mean,
    mode='lines+markers',
    line=dict(width=2, color='royalblue'),
    marker=dict(size=7, symbol='circle', color='royalblue')
))

fig.update_layout(
    template='plotly_white',
    title={
        "text": r"$\LARGE{\text{Comparaison de l'erreur moyenne en fonction du nombre d'itérations}} \\ \large{\text{Méthode itérative - 10 000 mesures par point - 360 prises de vues}}$"
    },
)

fig.update_xaxes(scaleanchor='x',
                 linewidth=1,
                 linecolor='black',
                 scaleratio=1,
                 showticklabels=True,
                 dtick=1,
                 )
fig.update_yaxes(scaleanchor='y',
                 linewidth=1,
                 linecolor='black',
                 showticklabels=True,
                 range=[0.0, 0.15]
                 )

fig.write_image(f"images/2 fig5 lr.webp", scale=1.3, height=600, width=1038)
fig.write_image(f"images/2 fig5 hr.png", scale=13, height=600, width=1038)

## NN


In [ ]:
import numpy as np

__import__("os").environ["CUDA_VISIBLE_DEVICES"] = ""


from tomography.neural_networks.direct_reconstruction import CTReconstructionNetwork

MODEL_DIRECT = CTReconstructionNetwork.load(
    "C:/Users/louis/Documents/Python/ADVANCED/Tomographie/tomography/neural_networks/models/Direct_0011.keras")
MODEL_DIRECT.projection_dim = 128
MODEL_DIRECT.latent_dim = 256

In [ ]:
for i, n_angle in enumerate([10, 25, 100, 360]):
    angles = np.linspace(0, 360, n_angle)
    sinogram = radon_transform(image, angles)
    print(sinogram.shape)
    reconstruction = MODEL_DIRECT.predict(sinogram.T, angles_deg=angles)

    fig = go.Figure(data=go.Heatmap(
        z=reconstruction,
        colorscale=COLORSCALE,
        showscale=False,
    ))

    fig.update_layout(
        template='plotly_white',
        title={
            "text": r"$\LARGE{\text{Reconstruction de la densité osseuse}} \\ \large{\text{Réseau de neurone - " + f"{n_angle} prises de vues" + "}} $"
        },
    )

    fig.update_xaxes(scaleanchor='x',
                     linewidth=1,
                     linecolor='black',
                     scaleratio=1,
                     showticklabels=False,
                     ticks=None, )
    fig.update_yaxes(scaleanchor='y',
                     linewidth=1,
                     linecolor='black',
                     showticklabels=False,
                     ticks=None,
                     )

    fig.write_image(f"images/5 fig5-{i} lr.webp", scale=1.3, height=600, width=600)
    fig.write_image(f"images/5 fig5-{i} hr.png", scale=13, height=600, width=600)


In [ ]:
from tomography.neural_networks.unet import CTUNet

MODEL = CTUNet.load(
    "C:/Users/louis/Documents/Python/ADVANCED/Tomographie/tomography/neural_networks/models/UNet_fft_0012.keras")
MODEL.bp_type = "fft"

for i, n_angle in enumerate([10, 25, 100, 360]):
    angles = np.linspace(0, 360, n_angle)
    sinogram = radon_transform(image, angles)
    print(sinogram.shape)
    reconstruction = MODEL.predict(sinogram.T, angles_deg=angles)

    fig = go.Figure(data=go.Heatmap(
        z=reconstruction,
        colorscale=COLORSCALE,
        showscale=False,
    ))

    fig.update_layout(
        template='plotly_white',
        title={
            "text": r"$\LARGE{\text{Reconstruction de la densité osseuse}} \\ \large{\text{Réseau de neurone - " + f"{n_angle} prises de vues" + "}} $"
        },
    )

    fig.update_xaxes(scaleanchor='x',
                     linewidth=1,
                     linecolor='black',
                     scaleratio=1,
                     showticklabels=False,
                     ticks=None, )
    fig.update_yaxes(scaleanchor='y',
                     linewidth=1,
                     linecolor='black',
                     showticklabels=False,
                     ticks=None,
                     )

    fig.write_image(f"images/5 fig6-{i} lr.webp", scale=1.3, height=600, width=600)
    fig.write_image(f"images/5 fig6-{i} hr.png", scale=13, height=600, width=600)
